# Exercício – Agentes Inteligentes: Aspirador Reflexo vs Agente Baseado em Utilidade

## Nome: Lucas Rohr Carreño

## Objetivo

O objetivo deste exercício é implementar e analisar diferentes tipos de agentes inteligentes no problema clássico do aspirador de pó, comparando:
Um agente reflexo simples
Um agente baseado em utilidade
Os alunos deverão analisar o desempenho dos agentes em múltiplas simulações e discutir eficiência e racionalidade.

### Parte 1 – Ambiente e Agente Reflexo
Considere o seguinte código inicial:

In [ ]:
import random

class AspiradorInteligenteReflexo:

    def __init__(self):
        # Ambiente 5x5
        # 0 = limpo
        # 1 = sujo
        self.ambiente = [[random.choice([0,1]) for _ in range(5)] for _ in range(5)]

        # posição inicial
        self.posicao = [0,0]

        # contador de ações
        self.acoes = 0

    def perceber(self):
        x,y = self.posicao
        return self.ambiente[x][y]

    def aspirar(self):
        x,y = self.posicao
        self.ambiente[x][y] = 0
        self.acoes += 1
        print(f"Aspirou sujeira em ({x}, {y})")

    def mover(self,direcao):

        if direcao == "direita" and self.posicao[1] < 4:
            self.posicao[1]+=1

        elif direcao == "esquerda" and self.posicao[1] > 0:
            self.posicao[1]-=1

        elif direcao == "baixo" and self.posicao[0] < 4:
            self.posicao[0]+=1

        elif direcao == "cima" and self.posicao[0] > 0:
            self.posicao[0]-=1

        self.acoes += 1
        print(f"Movido para {self.posicao}")

    def tudo_limpo(self):

        for linha in self.ambiente:
            for celula in linha:
                if celula == 1:
                    return False

        return True


    def executar(self,max_acoes=50):

        while not self.tudo_limpo() and self.acoes < max_acoes:

            if self.perceber()==1:
                self.aspirar()

            else:
                self.mover(random.choice(
                    ["direita","baixo","esquerda","cima"]
                ))
        print(f"Total de ações {self.acoes}")

# Executando a simulação
aspirador = AspiradorInteligenteReflexo()
aspirador.executar()

Aspirou sujeira em (0, 0)
Movido para [0, 0]
Movido para [0, 1]
Aspirou sujeira em (0, 1)
Movido para [1, 1]
Aspirou sujeira em (1, 1)
Movido para [1, 1]
Movido para [1, 0]
Aspirou sujeira em (1, 0)
Total de ações 9


### Questões – Parte 1

Após executar o programa algumas vezes, responda:

1. O agente sempre consegue limpar o ambiente? Explique.

- Não, houveram execuções onde o agente não se move ou ainda não se move para áreas sujas, pois o movimento é aleatório e não garante cobrir todas as áreas sujas.

2. Quantas ações foram necessárias em cada execução?

- Algumas 4 e outras 10, variando um tanto em pouco mais de 10 execuções que testei. Varia muito pois os movimentos são aleatórios e não existe certeza se as áreas sujas serão todas visitas e, se sim, quando.

3. O agente desperdiça ações? Explique quando isso acontece.

- Sim, ele desperdiça ações principalmente quando visita áreas que já estão limpas e ignora as sujas, pois não tem uma "inteligência" para isso. Além disso, ao tentar fazer um movimento inválido, o número de ações ainda é consumido (incrementado).

4. Qual o principal problema da estratégia utilizada?

- O principal seria não haver um tracking/memória de quais áreas são sujas e quais já foram ou estão limpas desde o começo. O agente é apenas reativo de acordo com a área aleatória que ele visita, o que torna ele pouco eficiente, sem uma rota otimizada.

5. Classifique o agente segundo os tipos vistos em aula, justificando sua resposta.

- É um Agente Reflexo Simples, pois ele apenas reage ao estado atual da área que está para tomar uma decisão (limpar ou não). Ele não aprende com um histórico e nem com o resto do ambiente para tomar uma decisão mais eficiente na limpeza.

Você pode modificar o código para facilitar a obtenção das resposta. Responda no próprio notebook, pois ele será entregue no Moodle posteriormente.

### Parte 2 – Agente Baseado em Utilidade
Agora você deve implementar um agente baseado em utilidade.

U=10×(salas limpas)−(acoes executadas)

Ou seja:
- limpar salas aumenta a utilidade
- movimentos desnecessários reduzem utilidade

O agente deve:
- evitar movimentos inválidos
- evitar revisitar salas já limpas
- priorizar salas ainda não visitadas

Requisitos:
- Aspirar sempre que encontrar sujeira
- Marcar salas visitadas
- Preferir salas não visitadas
- Parar quando tudo estiver limpo ou atingir max_acoes

In [ ]:
# Implemente seu código da parte 2 aqui. Execute considerando um máximo de 10 ações.
import random

class AspiradorInteligenteUtilidade:

    def __init__(self):
        # Ambiente 5x5
        # 0 = limpo
        # 1 = sujo
        self.ambiente = [[random.choice([0,1]) for _ in range(5)] for _ in range(5)]

        # matriz de visitadas
        # 0 = não visitada
        # 1 = visitada
        self.matriz_visitadas = [
            [0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0]
        ]

        # posição inicial
        self.posicao = [0, 0]

        # contador de ações
        self.acoes = 0

        # contador de salas limpas (para cálculo de utilidade)
        self.salas_limpas = 0

    def perceber(self):
        x, y = self.posicao
        return self.ambiente[x][y]

    def aspirar(self):
        x, y = self.posicao
        self.ambiente[x][y] = 0
        self.matriz_visitadas[x][y] = 1
        self.acoes += 1
        self.salas_limpas += 1
        print(f"Aspirou sujeira em ({x}, {y})")

    def calcular_utilidade(self):
        # U = 10 × (salas limpas) − (ações executadas)
        return 10 * self.salas_limpas - self.acoes

    def mover(self):
        x, y = self.posicao

        # Marca posição atual como visitada
        self.matriz_visitadas[x][y] = 1

        melhor_score = -float('inf')
        melhor_destino = None

        # Avalia cada vizinho pelo ganho de utilidade esperado, utilizando uma lista das posições dos vizinhos
        for nx, ny in [(x-1, y), (x+1, y), (x, y-1), (x, y+1)]:
            if 0 <= nx <= 4 and 0 <= ny <= 4: # se vizinho válido no ambiente
                if self.ambiente[nx][ny] == 1 and self.matriz_visitadas[nx][ny] == 0:
                    # Vizinho sujo não visitado: ganho de limpeza (+10) - custo mover (-1) - custo aspirar (-1)
                    score = 8
                elif self.matriz_visitadas[nx][ny] == 0:
                    # Vizinho limpo não visitado: apenas custo de mover, mas necessário para explorar
                    score = -1
                else:
                    # Vizinho já visitado: custo de mover sem ganho
                    score = -2

                if score > melhor_score:
                    melhor_score = score
                    melhor_destino = (nx, ny)

        # Só move se houver ganho esperado ou área não visitada ainda
        if melhor_destino is None or melhor_score <= -2:
            return  # Nenhum movimento útil, não consome ação

        # Caso contrário, encontrou melhor destino e se move para ele
        nx, ny = melhor_destino
        self.posicao = [nx, ny]
        self.acoes += 1
        print(f"Movido para {self.posicao}")

    def tudo_limpo(self):
        for linha in self.ambiente:
            for celula in linha:
                if celula == 1:
                    return False
        return True

    def executar(self, max_acoes=50):
        while not self.tudo_limpo() and self.acoes < max_acoes:
            if self.perceber() == 1:
                self.aspirar()
            else:
                self.mover()

        print(f"Total de ações: {self.acoes}")
        print(f"Salas limpas: {self.salas_limpas}")
        print(f"Utilidade final: U = 10×{self.salas_limpas} - {self.acoes} = {self.calcular_utilidade()}")


# Executando a simulação
aspirador = AspiradorInteligenteUtilidade()
aspirador.executar()

Movido para [1, 0]
Aspirou sujeira em (1, 0)
Total de ações: 2
Salas limpas: 1
Utilidade final: U = 10×1 - 2 = 8


### Parte 3 – Avaliação Experimental
Agora compare os dois agentes. Execute 100 simulações para cada agente.

O código abaixo possui uma implementação que pode ser utilizada como base. Modifique de acordo com a sua necessidade e, após, execute utilizando suas duas implementações como comparação, por exemplo. invocando:

experimento(AspiradorReflexo)
experimento(AspiradorUtilidade)

In [ ]:
def experimento(ClasseAgente):

    sucessos=0

    total_acoes=0

    execucoes=100

    for _ in range(execucoes):

        agente=ClasseAgente()

        agente.executar()

        total_acoes+=agente.acoes

        if agente.tudo_limpo():
            sucessos+=1


    print("Taxa de sucesso:",sucessos/execucoes)

    print("Ações médias:",total_acoes/execucoes)

experimento(AspiradorInteligenteReflexo)
experimento(AspiradorInteligenteUtilidade)

Movido para [1, 0]
Movido para [1, 1]
Movido para [1, 1]
Movido para [0, 1]
Aspirou sujeira em (0, 1)
Total de ações 5
Aspirou sujeira em (0, 0)
Movido para [0, 0]
Movido para [1, 0]
Movido para [1, 1]
Aspirou sujeira em (1, 1)
Movido para [1, 1]
Movido para [1, 1]
Movido para [1, 0]
Movido para [1, 0]
Movido para [1, 1]
Total de ações 10
Movido para [0, 1]
Aspirou sujeira em (0, 1)
Total de ações 2
Aspirou sujeira em (0, 0)
Movido para [0, 0]
Movido para [0, 1]
Aspirou sujeira em (0, 1)
Movido para [0, 1]
Movido para [0, 1]
Movido para [1, 1]
Movido para [1, 1]
Movido para [0, 1]
Movido para [0, 1]
Total de ações 10
Movido para [0, 0]
Movido para [1, 0]
Aspirou sujeira em (1, 0)
Total de ações 3
Aspirou sujeira em (0, 0)
Movido para [1, 0]
Movido para [1, 1]
Movido para [1, 1]
Movido para [1, 1]
Movido para [1, 0]
Movido para [1, 0]
Movido para [1, 0]
Movido para [1, 1]
Movido para [1, 0]
Total de ações 10
Movido para [0, 0]
Movido para [0, 0]
Movido para [1, 0]
Aspirou sujeira em (1,

### Questões – Parte 3

1. Qual agente teve maior taxa de sucesso? Explique o motivo.
- O agente baseado em utilidade teve a maior taxa de sucesso por justamente seguir uma lógica de priorizar sempre visitar áreas ainda não visitadas e que estejam sujas. Assim tornando a limpeza completa do ambiente algo controlado e não ocasional como no caso do agente de reflexo, que se preocupa apenas com o que "enxerga" em dada posição.
2. Qual agente utilizou menos ações em média? Explique o motivo.
- O agente baseado em utilidade utilizou menos ações em média, por conta de haver uma lógica que prioriza o aspirador se mover para áreas sujas e ainda não visitas, com o maior ganho esperado, o que diminui a possibilidade de visitar áreas já visitadas e/ou que já estão limpas. Isso acarreta em um desperdício menor de ações pelo agente de utilidade, diferente do de reflexo que se movimenta de forma aleatória sem prioridades, utilizando mais ações muitas vezes.
3. O agente baseado em utilidade sempre foi melhor? Explique situações onde ele pode não ser melhor.
- Em cenários onde todas as áreas já estavam limpas, acontece um empate de 0 movimentos entre ambos agentes. Já em casos de ambientes maiores, como 5x5, 10x10 e outros, o agente de utilidade performaria/escalaria bem mas não de forma ótima, pois tem uma lógica não tão rebuscada de "primeiro vizinho não visitado", o que pode não criar a melhor rota. O agente de reflexo aleatoriamente poderia ter uma chance muito pequena de cumprir melhor a limpeza em casos assim, mas é algo incerto e pouco provável.
4. Se o ambiente fosse 10×10 em vez de 2×2, o agente reflexo funcionaria bem? O agente baseado em utilidade escalaria melhor? Que modificações seriam necessárias?
- O agente de reflexo funcionaria muito mal pois com os movimentos aleatórios, as chances de completar a limpeza despencariam num ambiente grande. Já o agente de utilidade escalaria melhor, mas poderia não fazer as rotas mais bem otimizadas em questão de gasto de ações para efetuar a limpeza. Dentre modificações, além da matriz de ambiente, para garantir melho eficiência nas rotas, o agente de utilidade poderia usar um algoritmo de busca mais eficiente, como BFS (busca em largura), para encontrar as áreas sujas mais próximas e não necessariamente imediatas (adjacentes). Aumentar o máximo de ações para acompanhar o tamanho do ambiente também seria necessário para maior taxa de sucesso.

### Parte 4 - Ambiente maior

Modifique o ambiente para 5×5 e analise:

- O desempenho do agente reflexo mudou?
  - Teve uma alteração suave, se mantendo em média entre 0.5 e 0.6 (50% e 60%) de desempenho se olhando para a taxa de sucesso, assim como nos testes 2x2, podendo dizer que continou com um desempenho mediano em alguns casos e caiu em outros.
- O desempenho do agente de utilidade mudou?
  - Não mudou, continou com 100% de taxa de sucesso e com média de ações entre 3.5 e 4.0 como nos testes com ambiente 2x2, significando que escalou bem.
- Qual escalou melhor?
  - O agente de utilidade conseguiu manter uma taxa completa de 100% sem problemas e tendo pouquíssima variação na média de ações, o que indica que escalou melhor, diferente do de reflexo que teve decaídas, inclusive.
- Por quê isso aconteceu?
  - O mapeamento por ganho de utilidade escalou melhor justamente por buscar sempre as rotas otimizadas para a limpeza de áreas vizinhas ainda não visitadas e sujas, seguindo uma lógica ao invés de ser aleatório. Assim, mesmo com o aumento da área, o agente de utilidade não foi vítima da aleatoriedade e do acaso que o agente de reflexo foi, não desempenhando bem também no aumento do ambiente.